# Contrastive Language-Image Pretraining with SogCLR

### **Introduction**

In this tutorial, you will learn how to conduct contrastive language-image pretraining by optimizing the [Global Contrastive Loss](https://arxiv.org/abs/2202.12387) (GCL) on a subset of the [Conceptual Captions](https://ai.google.com/research/ConceptualCaptions/) dataset. Also, you will learn how to evaluate the model on retrieval task using the [MSCOCO](https://cocodataset.org/#home) dataset and zero-shot classification task using the [ImageNet](https://www.image-net.org/challenges/LSVRC/index.php) dataset. The code is based on [iSogCLR's](https://github.com/zhqiu/contrastive-learning-iSogCLR) codebase, which includes the implementation of CLIP, SogCLR and iSogCLR.

### Preparation

First, we:

1. Download the source code and data
2. Install required packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone -b project https://github.com/ShivaniSuresh1/csce689_iSogCLR.git iSogCLR

fatal: destination path 'iSogCLR' already exists and is not an empty directory.


In [ ]:
!cp /content/drive/Shareddrives/ML_Dataset/clip_train.tar.gz .
!cp /content/drive/Shareddrives/ML_Dataset/cc3m_subset_100k.tar.gz .
!cp /content/drive/Shareddrives/ML_Dataset/mscoco_val.tar.gz .
!cp /content/drive/Shareddrives/ML_Dataset/val.tar .

!mkdir -p datasets
!mkdir -p datasets/imagenet

!tar xf clip_train.tar.gz
!tar xf cc3m_subset_100k.tar.gz -C datasets
!tar xf mscoco_val.tar.gz -C datasets
!tar xf val.tar -C datasets/imagenet

!pip install -r ./iSogCLR/requirements_colab.txt



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 12.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 13.3 MB/s eta 0

In [ ]:
!git clone -b project https://github.com/ShivaniSuresh1/csce689_iSogCLR.git iSogCLR

!export PYTHONPATH="$PYTHONPATH:./iSogCLR/bimodal_exps"
!export HUGGINGFACE_HUB_CACHE='./checkpoints/huggingface'
!mkdir checkpoints

!gdown 142xxRoMaHxX3BIfCw_1b_G_dgu-02Yq3    # clip_train.tar.gz
# cc3m_subset_100k.tar.gz  (training images)
!gdown 1sFjRRqnSIl_QLnIPLY7-d3-fcyQshJkz

# ms_coco_val.tar.gz       (validation set for retrieval: image↔text)
!gdown 1QZMIl22EBA7bqJK5-oKSdXJI8sMMlZzt

# imagenet_val.tar         (validation set for zero-shot classification)
!gdown 1wPAfKltRYiJtyXLfAhtsifvRUf2ZNNke


!mkdir datasets
!mkdir -p datasets/imagenet
!tar xf clip_train.tar.gz
!tar xf cc3m_subset_100k.tar.gz -C datasets
!tar xf mscoco_val.tar.gz -C datasets
!tar xf val.tar -C datasets/imagenet

!pip install -r ./iSogCLR/requirements_colab.txt    # there may be pip warnings/ errors, should be fine to ignore them

fatal: destination path 'iSogCLR' already exists and is not an empty directory.
mkdir: cannot create directory ‘checkpoints’: File exists
Downloading...
From: https://drive.google.com/uc?id=142xxRoMaHxX3BIfCw_1b_G_dgu-02Yq3
To: /content/clip_train.tar.gz
100% 4.06M/4.06M [00:00<00:00, 279MB/s]
Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1sFjRRqnSIl_QLnIPLY7-d3-fcyQshJkz

but Gdown can't. Please check connections and permissions.
Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. 

### Training

The following command runs the training script to train a ResNet50 (pretrained on ImageNet) and a DistilBERT (pretrained on BookCorpus and English Wikipedia) on the cc3m dataset using the SogCLR loss for 30 epochs with temperature 0.01.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python ./iSogCLR/bimodal_exps/clip.py \
    --data_path ./datasets \
    --ann_path ./clip_train \
    --train_file cc3m_train_subset.json \
    --train_image_root cc3m_subset_100k \
    --output_dir output/isogclr_new_adafactor_e30\
    --init_model \
    --use_amp \
    --ita_type isogclr_new \
    --tau_init 0.02 \
    --sogclr_gamma 0.9 \
    --rho_I 8.0 \
    --rho_T 8.0 \
    --sched cosine \
    --opt adafactor \
    --lr 1e-4 \
    --no-distributed \
    --epochs 30 \
    --batch_size_train 128 \
    --isogclr_temp_net

2025-11-26 23:17:08.902277: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-26 23:17:08.919976: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764199028.941068    4074 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764199028.947531    4074 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764199028.964503    4074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

### Evaluation

The following command runs the evaluation script to evaluate the retrieval performance of the trained model on the MSCOCO validation dataset and the zero-shot classification performance on the ImageNet validation dataset. The evaluation command is obtained by appending `--evaluate --checkpoint /path/to/your/checkpoint --zs_dataset imagenet --zs_datafolder /path/to/imagenet/val` to the training command.

In [ ]:
CUDA_VISIBLE_DEVICES=3 python ./iSogCLR/bimodal_exps/clip.py \
    --data_path ./datasets \
    --ann_path ./clip_train \
    --train_file cc3m_train_subset.json \
    --train_image_root cc3m_subset_100k \
    --output_dir output/isogclr_new_adafactor_e30\
    --init_model \
    --use_amp \
    --ita_type isogclr_new \
    --tau_init 0.02 \
    --sogclr_gamma 0.9 \
    --rho_I 8.0 \
    --rho_T 8.0 \
    --sched cosine \
    --opt adafactor \
    --lr 1e-4 \
    --no-distributed \
    --epochs 30 \
    --batch_size_train 128 \
    --evaluate \
    --checkpoint 'output/isogclr_new_adafactor_e30/checkpoint_30.pth'\
    --zs_dataset imagenet \
    --zs_datafolder ./datasets/imagenet/val

### Benchmarks

The following results are recall at 1 results on the provided MSCOCO and ImageNet datasets. The first row of results are from the model trained using the CLIP loss, and the second row of results are from the model trained using the SogCLR loss. All results are based on a batch size of 128 for 30-epoch pretraining. IR@1 denotes the recall at 1 of image retrieval on MSCOCO, TR@1 denotes the recall at 1 of text retrieval on MSCOCO, and ACC@1 denotes the top 1 accuracy on ImageNet. Average denotes the average of the three metrics.

| Method | MSCOCO TR@1 | MSCOCO IR@1 | ImageNet ACC@1 | Average |
|:----------:|:--------:|:--------:|:--------:|:--------:|
| CLIP | 12.0 | 9.32 | 21.35 | 14.22 |
| SogCLR |  14.38  |  10.73  | 24.54 | 16.55 |

In [ ]:
import pandas as pd
data = [
    {"Method": "Adamp & CLIP",   "MSCOCO TR@1": 13.28, "MSCOCO IR@1": 9.296, "ImageNet ACC@1": 21.476},
    {"Method": "Adamp & iSogCLR", "MSCOCO TR@1": 9.78, "MSCOCO IR@1": 7.081, "ImageNet ACC@1": 19.56},
    {"Method": "Adamp & SogCLR",  "MSCOCO TR@1": 14.02, "MSCOCO IR@1": 10.24, "ImageNet ACC@1": 24.642},
    {"Method": "Adamp & SogCLR (Modified Parameters)", "MSCOCO TR@1": 12.74, "MSCOCO IR@1": 10.04, "ImageNet ACC@1": 24.86},
    {"Method": "Adamw & CyCLIP", "MSCOCO TR@1": 12.06, "MSCOCO IR@1": 9.24, "ImageNet ACC@1": 21.458},
    {"Method": "AdamW & iSogCLR_New_V2", "MSCOCO TR@1": 9.8, "MSCOCO IR@1": 7.51, "ImageNet ACC@1": 19.422},
    {"Method": "Nadam & iSogCLR_New_V2", "MSCOCO TR@1": 0.02, "MSCOCO IR@1": 0.02, "ImageNet ACC@1": 0.1},
    {"Method": "AdamW & SogCLR", "MSCOCO TR@1": 13.2, "MSCOCO IR@1": 9.91, "ImageNet ACC@1": 24.926},
    {"Method": "Adafactor & SogCLR", "MSCOCO TR@1": 13.68, "MSCOCO IR@1": 10.51, "ImageNet ACC@1": 25.288},
    {"Method": "AdamW & MarginHardNegCLIP", "MSCOCO TR@1": 12.56, "MSCOCO IR@1": 9.39, "ImageNet ACC@1": 22.76},
    {"Method": "Adam & HardNegCLIP", "MSCOCO TR@1": 12.0, "MSCOCO IR@1":9.30, "ImageNet ACC@1": 21.424},
    {"Method": "AdamW & iSogCLRv1", "MSCOCO TR@1": 0.02, "MSCOCO IR@1": 0.02, "ImageNet ACC@1": 0.1},
    {"Method": "Adam & SogCLR", "MSCOCO TR@1": 0, "MSCOCO IR@1": 0.02, "ImageNet ACC@1": 0.1},
    {"Method": "Adam & iSogCLRv2", "MSCOCO TR@1": 0.04, "MSCOCO IR@1":0.02, "ImageNet ACC@1": 0.1},
    {"Method": "Adafactor & SogCLR_Margin", "MSCOCO TR@1":13.54, "MSCOCO IR@1": 10.24, "ImageNet ACC@1": 25.158},
    {"Method": "Adafactor_Custom & SogCLR", "MSCOCO TR@1":13.04, "MSCOCO IR@1": 10.28, "ImageNet ACC@1": 24.632},
    {"Method": "Adamp and SogCLRMargin", "MSCOCO TR@1": 13.54, "MSCOCO IR@1": 10.64, "ImageNet ACC@1": 25.124},
    {"Method": "gcadamwp and SogCLR", "MSCOCO TR@1":0.02, "MSCOCO IR@1": 0.02, "ImageNet ACC@1": 0.1},
    {"Method": "AdafactorCustom & SogCLR_Margin", "MSCOCO TR@1":13.38, "MSCOCO IR@1": 9.90, "ImageNet ACC@1": 24.684},
    {"Method": "AdamW & SogCLR_Margin", "MSCOCO TR@1":12.92, "MSCOCO IR@1": 10.60, "ImageNet ACC@1": 24.78},
    {"Method": "Adafactor & iSogCLR_New", "MSCOCO TR@1":14.18, "MSCOCO IR@1": 10.28, "ImageNet ACC@1": 27.85},
    {"Method": "Adamp & iSogCLR_New", "MSCOCO TR@1":14.06, "MSCOCO IR@1": 9.99, "ImageNet ACC@1": 27.824},
    {"Method": "Adamw & iSogCLR_New", "MSCOCO TR@1":13.68, "MSCOCO IR@1": 10.34, "ImageNet ACC@1": 27.352},
    {"Method": "Adafactor & iSogCLR_New(gamma=0.95, lr = 1e-4)", "MSCOCO TR@1":14.5, "MSCOCO IR@1": 10.48, "ImageNet ACC@1": 29.32},
    {"Method": "Adamp & iSogCLR_New(gamma=0.95)", "MSCOCO TR@1":14.16, "MSCOCO IR@1": 10.424, "ImageNet ACC@1": 29.344},
    {"Method": "AdafactorCustom & iSogCLR_New", "MSCOCO TR@1":14.18, "MSCOCO IR@1": 10.676, "ImageNet ACC@1": 29.43},
    {"Method": "Adafactor (temp = 0.01 and gamma = 0.9) & iSogCLR_New", "MSCOCO TR@1":14.42, "MSCOCO IR@1": 10.748, "ImageNet ACC@1": 29.968},
    {"Method": "Adafactor (rho_I = 8, tau_unit=0.02, rho_T=8, learning rate = 1e-4 and gamma = 0.9) & iSogCLR_New", "MSCOCO TR@1":14.86, "MSCOCO IR@1": 10.916, "ImageNet ACC@1": 29.88},
    {"Method": "Adafactor (rho_I = 6, tau_unit=0.03, rho_T=6, learning rate = 1e-4 and gamma = 0.9) & iSogCLR_New", "MSCOCO TR@1":14, "MSCOCO IR@1": 10.576, "ImageNet ACC@1": 29.374},
    {"Method": "Adafactor (rho_I = 4, tau_unit=0.05, rho_T=12, learning rate = 5e-5 and gamma = 0.85) & iSogCLR_New", "MSCOCO TR@1":13.9, "MSCOCO IR@1": 11.119, "ImageNet ACC@1": 28.814},
    {"Method": "Adafactor (rho_I = 7, tau_unit=0.035, rho_T=9, learning rate = 8e-5 and gamma = 0.92) & iSogCLR_New", "MSCOCO TR@1":14.52, "MSCOCO IR@1": 10.832, "ImageNet ACC@1": 29.728},
    {"Method": "Adafactor_Custom (rho_I = 8, tau_unit=0.02, rho_T=8, learning rate = 1e-4 and gamma = 0.9) & iSogCLR_New", "MSCOCO TR@1":14.2, "MSCOCO IR@1": 10.816, "ImageNet ACC@1": 29.994}
]

df = pd.DataFrame(data)
df["Average"] = df[["MSCOCO TR@1", "MSCOCO IR@1", "ImageNet ACC@1"]].mean(axis=1)

print(df)

                                               Method  MSCOCO TR@1  \
0                                        Adamp & CLIP        13.28   
1                                     Adamp & iSogCLR         9.78   
2                                      Adamp & SogCLR        14.02   
3                Adamp & SogCLR (Modified Parameters)        12.74   
4                                      Adamw & CyCLIP        12.06   
5                              AdamW & iSogCLR_New_V2         9.80   
6                              Nadam & iSogCLR_New_V2         0.02   
7                                      AdamW & SogCLR        13.20   
8                                  Adafactor & SogCLR        13.68   
9                           AdamW & MarginHardNegCLIP        12.56   
10                                 Adam & HardNegCLIP        12.00   
11                                  AdamW & iSogCLRv1         0.02   
12                                      Adam & SogCLR         0.00   
13                  